
### BANKING CUSTOMER RESOLUTION SYSTEM

#### SECTION 1 — IMPORTS

In [6]:
from crewai import Agent, Task, Crew, Process, LLM
from langchain_ollama import ChatOllama

#### SECTION 2 - LOADING LLM

In [7]:
llm = LLM(
    model="ollama/mistral",
    base_url="http://localhost:11434"
)

#### SECTION 3 — DEFINING AGENTS

#### 1. Intent Classification Agent

In [8]:
intent_agent = Agent(
    
    role="Intent Classification Agent",

    goal="""
    Identify customer banking issue category and determine risk level.
    """,

    backstory="""
    Experienced banking analyst specialized in:
    - fraud detection
    - transaction failures
    - card issues
    - suspicious account activity
    - loan inquiries

    Responsible for identifying urgency and customer issue category.
    """,

    llm=llm,

    verbose=True
)

#### 2. Banking Policy Agent

In [9]:
policy_agent = Agent(

    role="Banking Rules & Policy Agent",

    goal="""
    Apply banking rules and determine valid resolution procedures.
    """,

    backstory="""
    Banking policy specialist with expertise in:
    - fraud handling
    - customer verification
    - transaction dispute policies
    - card replacement rules
    - banking compliance procedures

    Responsible for deciding the correct policy actions.
    """,

    llm=llm,

    verbose=True
)

#### 3. Response Generation Agent

In [10]:
response_agent = Agent(

    role="Response Generation Agent",

    goal="""
    Generate professional customer support responses.
    """,

    backstory="""
    Customer communication specialist trained in:
    - professional banking communication
    - customer interaction handling
    - explaining security procedures
    - resolution guidance

    Responsible for drafting customer-facing responses.
    """,

    llm=llm,

    verbose=True
)

#### 4. Risk & Escalation Agent

In [11]:
escalation_agent = Agent(

    role="Risk & Escalation Agent",

    goal="""
    Determine whether escalation is required.
    """,

    backstory="""
    Banking risk management specialist trained to identify:
    - fraud
    - suspicious activity
    - unauthorized access
    - high-risk financial incidents

    Responsible for escalation decisions and risk assessment.
    """,

    llm=llm,

    verbose=True
)

#### SECTION 4 — CUSTOMER QUERY

In [32]:
customer_query = """
My transaction failed but the amount was deducted from my account.
"""

#### SECTION 5 — TASKS

##### TASK 1 — INTENT CLASSIFICATION

In [33]:
classification_task = Task(

    description=f"""
    
    Analyze the following banking customer query:

    {customer_query}

    Identify:
    - issue category
    - urgency
    - fraud indicators
    - risk level

    Categories:
    - fraud
    - transaction
    - loan
    - card
    - general

    """,

    agent=intent_agent,

    expected_output="""
    Structured issue classification including:
    - category
    - urgency
    - fraud indicators
    - risk level
    """
)

##### TASK 2 — BANKING POLICY REASONING

In [34]:
policy_task = Task(

    description="""

    Using the classification result from the previous agent:

    Apply banking policies and determine:
    - verification requirements
    - account protection steps
    - customer resolution options
    - recommended next actions

    Banking Policies:

    1. Fraud or suspicious activity:
       - freeze account temporarily
       - escalate to fraud investigation team

    2. Failed transaction:
       - verify transaction details
       - initiate dispute process

    3. Card issues:
       - verify customer identity
       - process replacement request

    4. Loan inquiries:
       - provide loan eligibility and interest details

    """,

    agent=policy_agent,

    context=[classification_task],

    expected_output="""
    Banking policy reasoning and recommended actions.
    """
)

##### TASK 3 — RESPONSE GENERATION

In [35]:
response_task = Task(

    description="""

    Generate a professional customer response using:
    - issue classification
    - banking policy reasoning

    Requirements:
    - professional tone
    - clear explanation
    - mention next steps
    - mention security actions if applicable

    """,

    agent=response_agent,

    context=[
        classification_task,
        policy_task
    ],

    expected_output="""
    Professional customer support response.
    """
)

##### TASK 4 — ESCALATION DECISION

escalation_task = Task(

    description="""

    Determine whether this issue requires escalation.

    Escalate if:
    - fraud indicators detected
    - suspicious activity exists
    - unauthorized access suspected
    - high-value transaction involved
    - customer security compromised

    Otherwise:
    - mark issue as automatically resolvable

    Provide:
    - escalation decision
    - justification

    """,

    agent=escalation_agent,

    context=[
        classification_task,
        policy_task,
        response_task
    ],

    expected_output="""
    Final escalation decision with justification.
    """
)

### SECTION 6 — CREATING CREW

In [36]:
banking_crew = Crew(

    agents=[
        intent_agent,
        policy_agent,
        response_agent,
        escalation_agent
    ],

    tasks=[
        classification_task,
        policy_task,
        response_task,
        escalation_task
    ],

    process=Process.sequential,

    verbose=True
)

### SECTION 7 — RUNNING WORKFLOW

In [37]:
result = banking_crew.kickoff()

print("\n ----FINAL OUTPUT----\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: dd267746-2a0d-409d-baa1-bffd7ff6ebe7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│                                                                                                                 │
│      Analyze the following banking customer query:                                                              │
│                                                                                                                 │
│                                                                                                                 │
│  My transaction failed but the amount was deducted from my account.                                             │
│                                                                                                                 │
│                                                                                                                 │
│      Identify:                                                                                                  │
│      - issue category                                                                                           │
│      - urgency                                                                                                  │
│      - fraud indicators                                                                                         │
│      - risk level                                                                                               │
│                                                                                                                 │
│      Categories:                                                                                                │
│      - fraud                                                                                                    │
│      - transaction                                                                                              │
│      - loan                                                                                                     │
│      - card                                                                                                     │
│      - general                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  ID: 9177645b-a767-40a5-b26b-8c1275ecff20                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Intent Classification Agent                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│                                                                                                                 │
│      Analyze the following banking customer query:                                                              │
│                                                                                                                 │
│                                                                                                                 │
│  My transaction failed but the amount was deducted from my account.                                             │
│                                                                                                                 │
│                                                                                                                 │
│      Identify:                                                                                                  │
│      - issue category                                                                                           │
│      - urgency                                                                                                  │
│      - fraud indicators                                                                                         │
│      - risk level                                                                                               │
│                                                                                                                 │
│      Categories:                                                                                                │
│      - fraud                                                                                                    │
│      - transaction                                                                                              │
│      - loan                                                                                                     │
│      - card                                                                                                     │
│      - general                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Intent Classification Agent                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│   Category: Transaction                                                                                         │
│  Urgency: High                                                                                                  │
│  Fraud Indicators: Possible (a failed transaction that has deducted funds from the account is indicative of     │
│  potential fraudulent activity)                                                                                 │
│  Risk Level: Medium to High (due to the failure and unauthorized deduction, it is best to investigate further   │
│  to confirm whether the incident is a result of genuine technical issues or potential fraud)                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│      Analyze the following banking customer query:                                                              │
│                                                                                                                 │
│                                                                                                                 │
│  My transaction failed but the amount was deducted from my account.                                             │
│                                                                                                                 │
│                                                                                                                 │
│      Identify:                                                                                                  │
│      - issue category                                                                                           │
│      - urgency                                                                                                  │
│      - fraud indicators                                                                                         │
│      - risk level                                                                                               │
│                                                                                                                 │
│      Categories:                                                                                                │
│      - fraud                                                                                                    │
│      - transaction                                                                                              │
│      - loan                                                                                                     │
│      - card                                                                                                     │
│      - general                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Agent: Intent Classification Agent                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Risk & Escalation Agent                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│   Escalation Decision: Yes, this issue requires escalation due to the detection of fraud indicators in the      │
│  form of an unrecognized transaction of a significant amount ($5000).                                           │
│                                                                                                                 │
│  Justification: The current situation falls under the category of fraud with high urgency and risk level. As    │
│  per banking policies, a suspected fraud should be escalated immediately, and the customer's security must be   │
│  prioritized. In this case, the unauthorized transaction necessitates prompt action involving notifying the     │
│  customer, initiating necessary measures to secure their account, reporting the incident as fraud, and          │
│  conducting an independent investigation into the nature of the unauthorized activity. The ultimate goal is to  │
│  protect the customer's account while offering swift resolution during suspected incidents of fraud.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│                                                                                                                 │
│      Determine whether this issue requires escalation.                                                          │
│                                                                                                                 │
│      Escalate if:                                                                                               │
│      - fraud indicators detected                                                                                │
│      - suspicious activity exists                                                                               │
│      - unauthorized access suspected                                                                            │
│      - high-value transaction involved                                                                          │
│      - customer security compromised                                                                            │
│                                                                                                                 │
│      Otherwise:                                                                                                 │
│      - mark issue as automatically resolvable                                                                   │
│                                                                                                                 │
│      Provide:                                                                                                   │
│      - escalation decision                                                                                      │
│      - justification                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│  Agent: Risk & Escalation Agent                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


 ----FINAL OUTPUT----

 Escalation Decision: Yes, this issue requires escalation due to the detection of fraud indicators in the form of an unrecognized transaction of a significant amount ($5000).

Justification: The current situation falls under the category of fraud with high urgency and risk level. As per banking policies, a suspected fraud should be escalated immediately, and the customer's security must be prioritized. In this case, the unauthorized transaction necessitates prompt action involving notifying the customer, initiating necessary measures to secure their account, reporting the incident as fraud, and conducting an independent investigation into the nature of the unauthorized activity. The ultimate goal is to protect the customer's account while offering swift resolution during suspected incidents of fraud.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: dd267746-2a0d-409d-baa1-bffd7ff6ebe7                                                                       │
│  Final Output:  Escalation Decision: Yes, this issue requires escalation due to the detection of fraud          │
│  indicators in the form of an unrecognized transaction of a significant amount ($5000).                         │
│                                                                                                                 │
│  Justification: The current situation falls under the category of fraud with high urgency and risk level. As    │
│  per banking policies, a suspected fraud should be escalated immediately, and the customer's security must be   │
│  prioritized. In this case, the unauthorized transaction necessitates prompt action involving notifying the     │
│  customer, initiating necessary measures to secure their account, reporting the incident as fraud, and          │
│  conducting an independent investigation into the nature of the unauthorized activity. The ultimate goal is to  │
│  protect the customer's account while offering swift resolution during suspected incidents of fraud.            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯